# Tutorial 26: Streaming a LangGraph Run

Tutorials 7–25 build graphs that return one final state. A chat UI cannot wait for that. It needs tokens as they are generated, a status line while a tool runs, and a way to hide internal model calls from the user.

This tutorial streams one small support-desk graph. Every example runs offline with `GenericFakeChatModel`, which splits a canned reply into tokens. The last section swaps that model for Groq when `GROQ_API_KEY` is set.


## 1. What you can stream

`graph.stream()` and `graph.astream()` yield chunks while the graph runs. `stream_mode` chooses the payload:

| Mode | What you receive | Typical use |
|---|---|---|
| `updates` | The keys a node just wrote | Step list, "research finished" |
| `values` | The full state after each step | A client that replaces its copy of state |
| `messages` | LLM token chunks plus metadata | The text the user sees |
| `custom` | Dicts you emit with `get_stream_writer()` | Tool progress, status text |
| `checkpoints` | Same payload as `get_state()` | Debugging a thread |
| `tasks` | Task start and finish | Tracing parallel work |
| `debug` | Checkpoints, tasks, and extra metadata | A debugger pane |

Streaming does not replace `invoke()`. The graph still runs to completion and still returns state. The stream is a second view of that run.

Pass `version="v2"` (LangGraph 1.1+). Every chunk then has the same shape:

```python
{"type": "updates", "ns": (), "data": {...}, "interrupts": ()}
```

`ns` is `()` on the root graph and a tuple of node paths when the event comes from a subgraph. `interrupts` is how a human-in-the-loop pause shows up on the same stream (Tutorial 14).

Without `version="v2"`, a single mode returns raw data and several modes return `(mode, data)` tuples. The examples below use v2 so the client code stays the same when you add a mode.


## 2. Setup


In [1]:
from pathlib import Path
import os
from typing import TypedDict

from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.config import get_stream_writer
from langgraph.graph import END, START, StateGraph

try:
    from dotenv import load_dotenv
    for env_path in (Path.cwd() / ".env", Path.cwd().parent / ".env"):
        if env_path.exists():
            load_dotenv(env_path)
except ImportError:
    pass

print("Setup complete.")


Setup complete.


## 3. The graph we will stream

Three nodes, on purpose:

1. `research` calls a tool and emits custom progress.
2. `draft_reply` calls a public model. Its tokens belong on screen.
3. `score_reply` calls a second model tagged `nostream`. The score is stored in state, and its tokens are not streamed.

`get_stream_writer()` is safe to call even when the caller did not ask for `custom` mode. Those events are dropped.


In [2]:
class SupportState(TypedDict):
    topic: str
    policy: str
    reply: str
    score: str


@tool
def lookup_policy(topic: str) -> str:
    """Look up the support policy for a customer topic."""
    writer = get_stream_writer()
    writer({"kind": "tool", "name": "lookup_policy", "status": "searching"})
    return (
        f"Policy for {topic}: acknowledge the issue, give one next step, "
        "and keep the reply under two sentences."
    )


def build_support_graph(reply: str, score: str):
    """Return a new graph. The fake model reads each string once."""
    public_model = GenericFakeChatModel(messages=iter([reply]))
    private_model = GenericFakeChatModel(messages=iter([score])).with_config(
        {"tags": ["nostream"]}
    )

    def research(state: SupportState):
        writer = get_stream_writer()
        writer({"kind": "status", "text": "Checking the policy"})
        return {"policy": lookup_policy.invoke({"topic": state["topic"]})}

    def draft_reply(state: SupportState):
        writer = get_stream_writer()
        writer({"kind": "status", "text": "Drafting the customer reply"})
        message = public_model.invoke(
            [
                {
                    "role": "user",
                    "content": state["policy"] + " Topic: " + state["topic"],
                }
            ]
        )
        return {"reply": message.content}

    def score_reply(state: SupportState):
        message = private_model.invoke(
            [{"role": "user", "content": state["reply"]}]
        )
        return {"score": message.content}

    return (
        StateGraph(SupportState)
        .add_node(research)
        .add_node(draft_reply)
        .add_node(score_reply)
        .add_edge(START, "research")
        .add_edge("research", "draft_reply")
        .add_edge("draft_reply", "score_reply")
        .add_edge("score_reply", END)
        .compile()
    )


INPUTS = {"topic": "a late delivery", "policy": "", "reply": "", "score": ""}
REPLY = "Sorry your package is late. We reshipped it, and it arrives Thursday."
SCORE = "score=4. The reply names a date and a next step."

print("Graph factory ready.")


Graph factory ready.


## 4. `updates` and `values`

`updates` is the diff: the node name and only the keys that node returned. `values` is the whole state. The first chunk is the input; each later chunk is the state after a node, including keys written earlier.

Use `updates` to render a timeline. Use `values` when the client replaces its copy of the state on every chunk.


In [3]:
graph = build_support_graph(REPLY, SCORE)

print("updates")
for part in graph.stream(INPUTS, stream_mode="updates", version="v2"):
    node, wrote = next(iter(part["data"].items()))
    print(f"  {node}: {wrote}")


updates
  research: {'policy': 'Policy for a late delivery: acknowledge the issue, give one next step, and keep the reply under two sentences.'}
  draft_reply: {'reply': 'Sorry your package is late. We reshipped it, and it arrives Thursday.'}
  score_reply: {'score': 'score=4. The reply names a date and a next step.'}


In [4]:
graph = build_support_graph(REPLY, SCORE)

print("values")
for part in graph.stream(INPUTS, stream_mode="values", version="v2"):
    state = part["data"]
    print(
        f"  policy={bool(state['policy'])!s:5}  "
        f"reply={state['reply']!r}  score={state['score']!r}"
    )


values
  policy=False  reply=''  score=''
  policy=True   reply=''  score=''
  policy=True   reply='Sorry your package is late. We reshipped it, and it arrives Thursday.'  score=''
  policy=True   reply='Sorry your package is late. We reshipped it, and it arrives Thursday.'  score='score=4. The reply names a date and a next step.'


## 5. One chunk shape for every mode

v1 changes shape when you pass a list of modes. v2 does not. A client can branch on `part["type"]` and ignore modes it does not understand yet.


In [5]:
graph = build_support_graph(REPLY, SCORE)
v1 = next(graph.stream(INPUTS, stream_mode=["updates", "custom"]))

graph = build_support_graph(REPLY, SCORE)
v2 = next(graph.stream(INPUTS, stream_mode=["updates", "custom"], version="v2"))

print("v1 first chunk:", v1)
print("v2 first chunk:", v2)


v1 first chunk: ('custom', {'kind': 'status', 'text': 'Checking the policy'})
v2 first chunk: {'type': 'custom', 'ns': (), 'data': {'kind': 'status', 'text': 'Checking the policy'}}


## 6. Custom progress from a node and from a tool

Status text does not belong in the model stream. Emit it with `get_stream_writer()` and ask for `stream_mode="custom"`, usually together with `updates` or `messages`.

The tool above writes `{"kind": "tool", ...}` from inside `lookup_policy`. That works because the tool runs inside the graph, so it sees the same stream writer as the node.


In [6]:
graph = build_support_graph(REPLY, SCORE)

for part in graph.stream(INPUTS, stream_mode=["custom", "updates"], version="v2"):
    if part["type"] == "custom":
        print("custom ", part["data"])
    else:
        node = next(iter(part["data"]))
        print("update ", node)


custom  {'kind': 'status', 'text': 'Checking the policy'}
custom  {'kind': 'tool', 'name': 'lookup_policy', 'status': 'searching'}
update  research
custom  {'kind': 'status', 'text': 'Drafting the customer reply'}
update  draft_reply
update  score_reply


## 7. Tokens, and how to keep one model off the screen

`stream_mode="messages"` yields `(message_chunk, metadata)` for every LangChain chat model called inside the graph, including calls made with `.invoke()` rather than `.stream()`.

Metadata includes `langgraph_node` and any `tags` you set on the model. A model tagged `nostream` still runs and still writes state. Its tokens are omitted from `messages`.

Use that for a grader, a router, or a summariser whose text would otherwise flicker through the chat transcript.


In [7]:
graph = build_support_graph(REPLY, SCORE)
seen_nodes = set()

for part in graph.stream(
    INPUTS, stream_mode=["messages", "updates"], version="v2"
):
    if part["type"] == "messages":
        chunk, metadata = part["data"]
        if chunk.content:
            seen_nodes.add(metadata["langgraph_node"])
            print(chunk.content, end="")
    elif "score_reply" in part["data"]:
        print()
        print("score in state:", part["data"]["score_reply"]["score"])

print("nodes that streamed tokens:", sorted(seen_nodes))


Sorry your package is late. We reshipped it, and it arrives Thursday.
score in state: score=4. The reply names a date and a next step.
nodes that streamed tokens: ['draft_reply']


## 8. Filter tokens by node

`nostream` drops a call entirely. Sometimes you still want both calls in the stream, and the UI only prints one of them — a translation beside the draft, for example. Filter on `metadata["langgraph_node"]`.


In [8]:
class DraftState(TypedDict):
    topic: str
    reply: str
    translation: str


def build_two_writers():
    reply_model = GenericFakeChatModel(messages=iter([
        "We reshipped the package. It arrives Thursday."
    ]))
    translation_model = GenericFakeChatModel(messages=iter([
        "Abbiamo rispedito il pacco. Arriva giovedi."
    ]))

    def draft(state: DraftState):
        message = reply_model.invoke(
            [{"role": "user", "content": state["topic"]}]
        )
        return {"reply": message.content}

    def translate(state: DraftState):
        message = translation_model.invoke(
            [{"role": "user", "content": state["reply"]}]
        )
        return {"translation": message.content}

    return (
        StateGraph(DraftState)
        .add_node(draft)
        .add_node(translate)
        .add_edge(START, "draft")
        .add_edge("draft", "translate")
        .add_edge("translate", END)
        .compile()
    )


graph = build_two_writers()
print("customer-facing tokens only:")
for part in graph.stream(
    {"topic": "late delivery", "reply": "", "translation": ""},
    stream_mode="messages",
    version="v2",
):
    chunk, metadata = part["data"]
    if chunk.content and metadata["langgraph_node"] == "draft":
        print(chunk.content, end="")
print()


customer-facing tokens only:
We reshipped the package. It arrives Thursday.


## 9. Subgraphs

A compiled graph used as a node is a subgraph. With `subgraphs=False` (the default), the parent stream shows the subgraph as one update and hides everything inside it.

That includes tokens. `create_agent()` returns a compiled graph, so adding it as a node and then streaming the parent with `stream_mode="messages"` produces no tokens until you pass `subgraphs=True`. Those events carry a non-empty `ns` tuple, such as `("child:<task_id>",)`.

The same flag applies to every mode, not only `messages`.


In [9]:
class ChildState(TypedDict):
    foo: str
    bar: str


def expand(state: ChildState):
    return {"bar": "bar"}


def join(state: ChildState):
    return {"foo": state["foo"] + state["bar"]}


child = (
    StateGraph(ChildState)
    .add_node(expand)
    .add_node(join)
    .add_edge(START, "expand")
    .add_edge("expand", "join")
    .compile()
)


class ParentState(TypedDict):
    foo: str


def greet(state: ParentState):
    return {"foo": "hi! " + state["foo"]}


parent = (
    StateGraph(ParentState)
    .add_node("greet", greet)
    .add_node("child", child)
    .add_edge(START, "greet")
    .add_edge("greet", "child")
    .compile()
)

print("parent only")
for part in parent.stream({"foo": "foo"}, stream_mode="updates", version="v2"):
    print(" ", next(iter(part["data"])))

print("parent and child")
for part in parent.stream(
    {"foo": "foo"}, stream_mode="updates", subgraphs=True, version="v2"
):
    where = "child" if part["ns"] else "root"
    node, wrote = next(iter(part["data"].items()))
    print(f"  {where:5} {node}: {wrote}")


parent only
  greet
  child
parent and child
  root  greet: {'foo': 'hi! foo'}
  child expand: {'bar': 'bar'}
  child join: {'foo': 'hi! foobar'}
  root  child: {'foo': 'hi! foobar'}


## 10. Checkpoints on the same stream

`checkpoints` needs a checkpointer, the same kind Tutorial 19 uses for memory. Each chunk matches `get_state()`: `values`, `next`, and `metadata`. This is for a debugger, not for the chat transcript. `tasks` and `debug` follow the same rule.


In [10]:
class EchoState(TypedDict):
    text: str


def shout(state: EchoState):
    return {"text": state["text"].upper()}


checked = (
    StateGraph(EchoState)
    .add_node(shout)
    .add_edge(START, "shout")
    .compile(checkpointer=MemorySaver())
)

config = {"configurable": {"thread_id": "tutorial-26"}}
for part in checked.stream(
    {"text": "saved"},
    config,
    stream_mode="checkpoints",
    version="v2",
):
    snapshot = part["data"]
    step = snapshot["metadata"]["step"]
    print(f"step {step:>2}  next={list(snapshot['next'])}  values={snapshot['values']}")


step -1  next=['__start__']  values={}
step  0  next=['shout']  values={'text': 'saved'}
step  1  next=[]  values={'text': 'SAVED'}


## 11. A terminal UI

A client reads one loop. Custom events become the status line, message chunks become the reply, and `updates` is how you know a step finished. `astream` is the same loop with `async for`, which is what a FastAPI or WebSocket handler uses (Tutorial 13).


In [11]:
def render(graph, inputs):
    status = ""
    reply = []
    for part in graph.stream(
        inputs,
        stream_mode=["custom", "messages", "updates"],
        version="v2",
    ):
        if part["type"] == "custom" and part["data"].get("kind") == "status":
            status = part["data"]["text"]
            print()
            print(f"[{status}]")
        elif part["type"] == "messages":
            chunk, metadata = part["data"]
            if chunk.content and metadata["langgraph_node"] == "draft_reply":
                reply.append(chunk.content)
                print(chunk.content, end="")
        elif part["type"] == "updates" and "score_reply" in part["data"]:
            print()
            print()
            print(
                "internal score kept off screen:",
                part["data"]["score_reply"]["score"],
            )
    return "".join(reply)


graph = build_support_graph(REPLY, SCORE)
final_reply = render(graph, INPUTS)
assert final_reply == REPLY
print()
print("rendered reply matches the public model.")



[Checking the policy]

[Drafting the customer reply]
Sorry your package is late. We reshipped it, and it arrives Thursday.

internal score kept off screen: score=4. The reply names a date and a next step.

rendered reply matches the public model.


## 12. Optional: the same loop with Groq

The cells above do not need an API key. Set `GROQ_API_KEY` and rerun this cell to see real tokens. The stream loop does not change — only the model does.


In [12]:
if not os.getenv("GROQ_API_KEY"):
    print("GROQ_API_KEY is not set. Skipping the live model.")
else:
    from langchain.chat_models import init_chat_model

    live = init_chat_model("groq:llama-3.1-8b-instant", temperature=0)

    class LiveState(TypedDict):
        topic: str
        reply: str

    def answer(state: LiveState):
        message = live.invoke(
            [{"role": "user", "content": "Reply in one sentence about " + state["topic"] + "."}]
        )
        return {"reply": message.content}

    live_graph = (
        StateGraph(LiveState)
        .add_node(answer)
        .add_edge(START, "answer")
        .add_edge("answer", END)
        .compile()
    )
    print("live tokens:")
    for part in live_graph.stream(
        {"topic": "parcel tracking", "reply": ""},
        stream_mode="messages",
        version="v2",
    ):
        chunk, _metadata = part["data"]
        if chunk.content:
            print(chunk.content, end="")
    print()


GROQ_API_KEY is not set. Skipping the live model.


## 13. Summary

- Stream with `version="v2"` so every chunk is `type`, `ns`, `data`.
- `updates` for a timeline, `values` for a full state snapshot, `messages` for tokens, `custom` for progress you write yourself.
- Tag internal model calls with `nostream`. Filter `langgraph_node` when several public calls share one stream.
- Pass `subgraphs=True` when the model runs inside a nested graph, including `create_agent()` used as a node.
- `checkpoints`, `tasks`, and `debug` need a checkpointer and are for debugging a thread.

Next steps:

1. Drive this loop from the FastAPI app in Tutorial 13.
2. Surface `part["interrupts"]` with the approval flow in Tutorial 14.
3. Stream a Deep Agent (Tutorial 22) the same way. It is a compiled graph.
